# 02 — Render Sage spherical sequences with Go

This notebook reads `sage-spherical-sequences-for-go.json`, validates all
eight sequences, plots `log10(|value|)` versus order, saves a PNG, and
displays it inline.

Run notebook 01 first. Select the **Go (gonb)** kernel and run all cells.


In [ ]:
import (
    "encoding/json"
    "fmt"
    "image"
    "image/color"
    "image/draw"
    "image/png"
    "math"
    "os"
    "sort"

    "github.com/janpfeifer/gonb/gonbui"
)

type Dataset struct {
    Schema string
    MaxL, Width, Height, WorkingBits int
    BoundaryArguments map[string][2]float64
    Sequences map[string][][2]float64
}

var palette = []color.RGBA{
    {31,119,180,255}, {255,127,14,255}, {44,160,44,255}, {214,39,40,255},
    {148,103,189,255}, {140,86,75,255}, {227,119,194,255}, {23,190,207,255},
}

func finite(x float64) bool { return !math.IsNaN(x) && !math.IsInf(x, 0) }
func magnitude(z [2]float64) float64 { return math.Hypot(z[0], z[1]) }
func line(img *image.RGBA, x0,y0,x1,y1 int, c color.RGBA) {
    dx,dy:=int(math.Abs(float64(x1-x0))),int(math.Abs(float64(y1-y0)))
    sx,sy:=-1,-1; if x0<x1 { sx=1 }; if y0<y1 { sy=1 }
    err:=dx-dy
    for {
        for ox:=-1; ox<=1; ox++ { for oy:=-1; oy<=1; oy++ { img.SetRGBA(x0+ox,y0+oy,c) } }
        if x0==x1 && y0==y1 { break }
        e2:=2*err; if e2 > -dy { err-=dy; x0+=sx }; if e2 < dx { err+=dx; y0+=sy }
    }
}

func render() error {
    raw,err:=os.ReadFile("sage-spherical-sequences-for-go.json"); if err!=nil{return err}
    var data Dataset; if err=json.Unmarshal(raw,&data); err!=nil{return err}
    if data.Schema!="sage-spherical-sequences-for-go-v1" { return fmt.Errorf("unexpected schema %q",data.Schema) }
    if data.Width!=800||data.Height!=800||data.MaxL!=42 { return fmt.Errorf("expected full_800 profile, got %dx%d L=%d",data.Width,data.Height,data.MaxL) }
    names:=make([]string,0,len(data.Sequences)); total:=0
    lo,hi:=math.Inf(1),math.Inf(-1)
    for name,values:=range data.Sequences {
        if len(values)!=data.MaxL+1 { return fmt.Errorf("%s has %d values",name,len(values)) }
        names=append(names,name); total+=len(values)
        for _,z:=range values {
            if !finite(z[0])||!finite(z[1]) { return fmt.Errorf("non-finite value in %s",name) }
            y:=math.Log10(math.Max(magnitude(z),1e-300)); if y<lo{lo=y}; if y>hi{hi=y}
        }
    }
    sort.Strings(names); if len(names)!=8||total!=344{return fmt.Errorf("expected 8 x 43, got %d and %d",len(names),total)}
    W,H:=data.Width,data.Height; img:=image.NewRGBA(image.Rect(0,0,W,H))
    draw.Draw(img,img.Bounds(),&image.Uniform{color.RGBA{250,250,252,255}},image.Point{},draw.Src)
    left,right,top,bottom:=80,W-30,30,H-70
    axis:=color.RGBA{30,30,30,255}; line(img,left,top,left,bottom,axis); line(img,left,bottom,right,bottom,axis)
    for i,name:=range names {
        values:=data.Sequences[name]
        for ell:=1; ell<len(values); ell++ {
            project:=func(k int)(int,int){
                x:=left+int(float64(k)*float64(right-left)/float64(data.MaxL))
                v:=math.Log10(math.Max(magnitude(values[k]),1e-300))
                y:=bottom-int((v-lo)/(hi-lo)*float64(bottom-top)); return x,y
            }
            x0,y0:=project(ell-1); x1,y1:=project(ell); line(img,x0,y0,x1,y1,palette[i])
        }
    }
    out,err:=os.Create("sage-spherical-sequences-go-render.png"); if err!=nil{return err}
    if err=png.Encode(out,img); err!=nil{out.Close();return err}; if err=out.Close();err!=nil{return err}
    fmt.Printf("schema=%s; MaxL=%d; sequences=%d; complex values=%d\n",data.Schema,data.MaxL,len(names),total)
    fmt.Printf("log10 magnitude range: [%.6g, %.6g]\n",lo,hi)
    fmt.Println("series order/colors:"); for i,name:=range names { fmt.Printf("  %s -> RGB(%d,%d,%d)\n",name,palette[i].R,palette[i].G,palette[i].B) }
    gonbui.DisplayMarkdown("### Sage certified spherical sequences — `log10(|value|)` by order")
    return gonbui.DisplayImage(img)
}

In [ ]:
%%
if err := render(); err != nil {
    panic(err)
}
fmt.Println("GO JSON VALIDATION AND RENDER: PASS")